# Colab-first road flood workflow
Run cells top to bottom in a fresh Colab runtime. Mount Google Drive using Colab's Files panel first (at `/content/drive`). The same standard-Python cells also work in a Jupyter kernel on a Linux `/content` host. Project packages run only in uv's isolated Python environment.

Prepare Drive `raw/` with the exact tracked source-manifest files. Optional human-reviewed CSVs belong in `review/review_decisions.csv` and `review/event_reviews.csv`. Outputs go to `runs/<run_id>/`. Keep the raw/review snapshot unchanged through local verification. Set a full source SHA and a fresh run ID below; public cloning needs no credentials. No raw files or review contents are embedded in this template.


In [ ]:
import hashlib
import json
import os
import re
import shutil
import subprocess
import urllib.request
from datetime import UTC, datetime
from pathlib import Path

REPO_URL = "https://github.com/duygri/hcmc-road-flood-ml.git"
SOURCE_COMMIT = ""  # Required: full 40-character Git commit SHA.
RUN_ID = ""  # Required: a fresh run identifier.
DRIVE_ROOT = Path("/content/drive/MyDrive/hcmc-road-flood-ml")
PUBLISH_INTERIM = False
OUTPUT_RESERVE_GIB = 20  # Increase for the selected AOI and intermediates.
CHECKOUT = Path("/content/hcmc-road-flood-ml")

if not re.fullmatch(r"[0-9a-fA-F]{40}", SOURCE_COMMIT):
    raise ValueError("Set SOURCE_COMMIT to the full 40-character SHA")
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.-]*", RUN_ID) or RUN_ID in {".", ".."}:
    raise ValueError("Set a safe, fresh RUN_ID")
if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Mount Google Drive before continuing")
DRIVE_ROOT = DRIVE_ROOT.resolve()
RAW_ROOT = (DRIVE_ROOT / "raw").resolve()
if not RAW_ROOT.is_dir() or not RAW_ROOT.is_relative_to(DRIVE_ROOT):
    raise ValueError("Drive raw/ must exist within DRIVE_ROOT")
if (DRIVE_ROOT / "runs" / RUN_ID).exists():
    raise FileExistsError("Choose a new RUN_ID; a published bundle already exists")
if CHECKOUT.exists():
    raise FileExistsError("Use a fresh runtime or a new empty CHECKOUT path")
if OUTPUT_RESERVE_GIB < 0:
    raise ValueError("OUTPUT_RESERVE_GIB must be nonnegative")
STARTED_AT = datetime.now(UTC).isoformat()

## Standalone uv and pinned checkout
The versioned Astral installer installs a user-owned standalone binary, without sudo or modifying the Colab kernel. Python 3.12 and all project packages live under uv management. See [uv installation](https://docs.astral.sh/uv/getting-started/installation/).


In [ ]:
UV_BIN_DIR = Path("/content/uv-bin")
ENV = dict(
    os.environ,
    UV_INSTALL_DIR=str(UV_BIN_DIR),
    UV_NO_MODIFY_PATH="1",
    UV_PROJECT_ENVIRONMENT="/content/.venv",
    UV_PYTHON="3.12",
)
with urllib.request.urlopen("https://astral.sh/uv/0.13.6/install.sh") as response:
    installer = response.read()
subprocess.run(["sh"], input=installer, env=ENV, check=True)
UV = str(UV_BIN_DIR / "uv")


def require_uv_version(output, expected="0.13.6"):
    match = re.match(r"^uv\s+(\d+\.\d+\.\d+)(?:\s|$)", output.strip())
    if not match or match.group(1) != expected:
        raise RuntimeError(f"Unexpected uv version: {output.strip()}")
    return output.strip()  # Keep build/platform metadata for provenance.


UV_VERSION = require_uv_version(
    subprocess.run([UV, "--version"], check=True, capture_output=True, text=True).stdout
)
subprocess.run([UV, "python", "install", "3.12"], env=ENV, check=True)
subprocess.run(["git", "clone", REPO_URL, str(CHECKOUT)], check=True)
subprocess.run(["git", "checkout", "--detach", SOURCE_COMMIT], cwd=CHECKOUT, check=True)
actual_commit = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=CHECKOUT,
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
if actual_commit.lower() != SOURCE_COMMIT.lower():
    raise RuntimeError("Checkout does not match SOURCE_COMMIT")
subprocess.run(
    [UV, "sync", "--project", str(CHECKOUT), "--locked", "--group", "dev"],
    cwd=CHECKOUT,
    env=ENV,
    check=True,
)
REPORTS = CHECKOUT / "reports" / "runs" / RUN_ID
if REPORTS.exists():
    raise FileExistsError("Run output already exists")
REPORTS.mkdir(parents=True)


def run_project(args, name):
    """Capture every project command in the selected run's report directory."""
    command = [UV, "run", "--project", str(CHECKOUT), "--locked", *map(str, args)]
    result = subprocess.run(
        command, check=False, cwd=CHECKOUT, env=ENV, capture_output=True, text=True
    )
    (REPORTS / f"{name}.stdout.log").write_text(result.stdout, encoding="utf-8")
    (REPORTS / f"{name}.stderr.log").write_text(result.stderr, encoding="utf-8")
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()
    return result


COMMON = [
    "--config",
    "configs/experiments.yaml",
    "--repo-root",
    str(CHECKOUT),
    "--run-id",
    RUN_ID,
]

## Manifest verification, scratch preflight, and optional reviews
Every manifest asset must match its size and SHA-256, including supporting documents. Scratch capacity must cover the manifest input sizes plus the configurable output reserve. PyYAML is imported only in the isolated subprocess. Missing reviews stay missing: no labels are manufactured, and Track B can report `pending_positive_review`.


In [ ]:
preflight = r"""
import hashlib
import json
import shutil
import sys
from pathlib import Path
import yaml
root = Path(sys.argv[1]).resolve()
manifest = yaml.safe_load(Path("data/raw/manifest.yaml").read_text(encoding="utf-8"))
assets = [asset for dataset in manifest["datasets"] for asset in dataset["assets"]]
required = sum(int(asset["bytes"]) for asset in assets) + int(float(sys.argv[2]) * 1024**3)
if shutil.disk_usage("/content").free < required:
    raise RuntimeError(f"Insufficient /content scratch space: need {required} bytes")
verified = {}
for asset in assets:
    path = (root / asset["path"]).resolve()
    if not path.is_relative_to(root) or not path.is_file():
        raise ValueError(f"Missing or unsafe manifest asset: {asset['path']}")
    if path.stat().st_size != int(asset["bytes"]):
        raise ValueError(f"Size mismatch: {asset['path']}")
    with path.open("rb") as stream:
        digest = hashlib.file_digest(stream, "sha256").hexdigest()
    if digest != asset["sha256"]:
        raise ValueError(f"SHA-256 mismatch: {asset['path']}")
    verified[asset["path"]] = digest
print(json.dumps({"input_hashes": verified, "required_scratch_bytes": required}))
"""
run_project(
    ["python", "-c", preflight, str(RAW_ROOT), str(OUTPUT_RESERVE_GIB)], "preflight"
)
LOCAL = CHECKOUT / "configs" / "local"
LOCAL.mkdir(parents=True, exist_ok=True)
# JSON is a valid YAML subset; write only the supported local override.
(LOCAL / "sources.yaml").write_text(
    json.dumps({"sources": {"raw_root": str(RAW_ROOT)}}), encoding="utf-8"
)
REVIEW_ARGS = []
review_hashes = {}
for filename, flag in (
    ("review_decisions.csv", "--match-reviews"),
    ("event_reviews.csv", "--event-reviews"),
):
    review = DRIVE_ROOT / "review" / filename
    if review.is_file():
        destination = LOCAL / filename
        shutil.copyfile(review, destination)
        with destination.open("rb") as stream:
            review_hashes[filename] = hashlib.file_digest(stream, "sha256").hexdigest()
        REVIEW_ARGS.extend([flag, f"configs/local/{filename}"])
    else:
        print(
            f"No {filename}; retaining no-review setup (pending_positive_review when positives are absent)"
        )

## Synthetic smoke test before raw execution
This test uses synthetic fixtures. A failure stops execution before the pipeline touches raw inputs.


In [ ]:
run_project(["pytest", "tests/integration/test_track_b.py", "-q"], "synthetic_smoke")

## Pipeline
Build derived inputs and the baseline with the present human reviews only.


In [ ]:
run_project(["python", "scripts/run_pipeline.py", *COMMON, *REVIEW_ARGS], "pipeline")

## Track B fit gate
Train against confirmed event links and the frozen partitions. A pending fit gate is a reportable outcome.


In [ ]:
run_project(
    [
        "python",
        "scripts/train.py",
        *COMMON,
        "--features",
        f"data/processed/{RUN_ID}/road_features.parquet",
        "--labels",
        f"data/processed/{RUN_ID}/road_labels.parquet",
        "--confirmed-event-segments",
        f"data/processed/{RUN_ID}/confirmed_event_segments.parquet",
        "--partitions",
        f"data/processed/{RUN_ID}/road_partitions.parquet",
    ],
    "train",
)

## Evaluation
Only a `ready` Track B report permits the Track B scores argument. Pending gates still evaluate and publish the baseline report.


In [ ]:
track_b = json.loads((REPORTS / "track_b.json").read_text(encoding="utf-8"))
print("Track B:", track_b["status"])
evaluation_args = [
    "python",
    "scripts/evaluate.py",
    *COMMON,
    "--scores",
    f"data/processed/{RUN_ID}/road_susceptibility.parquet",
    "--roads",
    f"data/interim/{RUN_ID}/road_segments.gpkg",
    "--confirmed-event-segments",
    f"data/processed/{RUN_ID}/confirmed_event_segments.parquet",
    "--partitions",
    f"data/processed/{RUN_ID}/road_partitions.parquet",
]
if track_b["status"] == "ready":
    evaluation_args.extend(
        ["--track-b-scores", f"data/processed/{RUN_ID}/track_b_scores.parquet"]
    )
run_project(evaluation_args, "evaluate")

## Runtime, summary, and verified Drive publication
The publisher verifies copied hashes and writes its completion manifest last. A failed command stops publication. Inspect the published manifest before syncing the result to Windows.


In [ ]:
runtime_result = run_project(
    [
        "python",
        "-c",
        "import json, platform; print(json.dumps({'python_version': platform.python_version(), 'platform': platform.platform()}))",
    ],
    "runtime",
)
runtime = json.loads(runtime_result.stdout)
runtime.update(
    {
        "uv_version": UV_VERSION,
        "source_commit": actual_commit,
        "run_id": RUN_ID,
        "started_at": STARTED_AT,
        "finished_at": datetime.now(UTC).isoformat(),
        "review_hashes": review_hashes,
        "output_reserve_gib": OUTPUT_RESERVE_GIB,
    }
)
(REPORTS / "runtime.json").write_text(
    json.dumps(runtime, indent=2) + "\n", encoding="utf-8"
)
evaluation = json.loads((REPORTS / "evaluation.json").read_text(encoding="utf-8"))
summary = (
    f"# Colab run {RUN_ID}\n\nSource: `{actual_commit}`\n\n"
    f"Track B: `{track_b['status']}`\n\nEvaluation: `{evaluation['status']}`\n\n"
    f"uv: `{UV_VERSION}`; Python: `{runtime['python_version']}`\n\n"
    f"Review snapshot hashes: `{json.dumps(review_hashes, sort_keys=True)}`\n\n"
    "Input verification and command stdout/stderr accompany this report.\n"
)
(REPORTS / "colab_summary.md").write_text(summary, encoding="utf-8")
publish_args = [
    "python",
    "scripts/publish_run.py",
    *COMMON,
    "--drive-root",
    str(DRIVE_ROOT),
]
if PUBLISH_INTERIM:
    publish_args.append("--publish-interim")
# Write publisher logs locally only after publication completes. They cannot be
# part of the bundle whose immutable completion manifest the publisher just wrote.
# Earlier command logs and the summary are included in that verified bundle.
publication = subprocess.run(
    [UV, "run", "--project", str(CHECKOUT), "--locked", *publish_args],
    check=False,
    cwd=CHECKOUT,
    env=ENV,
    capture_output=True,
    text=True,
)
(REPORTS / "publish.stdout.log").write_text(publication.stdout, encoding="utf-8")
(REPORTS / "publish.stderr.log").write_text(publication.stderr, encoding="utf-8")
print(publication.stdout)
print(publication.stderr)
publication.check_returncode()
print(summary)
print("Published bundle:", DRIVE_ROOT / "runs" / RUN_ID)

## Windows/local replay in a separate checkout
Install standalone uv 0.13.6 locally. Sync the complete published `runs/<run_id>/` bundle and the **same raw/review snapshot** from Drive first. Preserve absent review files as absent. Do not copy Colab derived outputs into the replay checkout. Keep the Colab result unchanged; the verifier compares already-produced outputs and does not launch replay.

Run the following standard Python script locally (for example using `py replay.py` on Windows). Replace the SHA, run ID, and paths. `LOCAL_SNAPSHOT` points to the synced Drive root; `REPLAY` must be a new, separate checkout. Match any changed notebook parameters and retain the raw snapshot until verification is complete. The replay script verifies the synced raw hashes and preflights the local scratch volume before execution. It also creates local runtime, summary, and command logs required by the verifier; none are copied from the Colab result.

```python
from pathlib import Path
import json
import os
import re
import shutil
import subprocess

REPO_URL = "https://github.com/duygri/hcmc-road-flood-ml.git"
SOURCE_COMMIT = "REPLACE_WITH_THE_SAME_FULL_40_CHARACTER_SHA"
RUN_ID = "REPLACE_WITH_THE_SAME_RUN_ID"
LOCAL_SNAPSHOT = Path("drive-snapshot").resolve()
REPLAY = Path("local-replay").resolve()  # separate checkout, never the published bundle
assert not REPLAY.exists()
def require_uv_version(output, expected="0.13.6"):
    match = re.match(r"^uv\s+(\d+\.\d+\.\d+)(?:\s|$)", output.strip())
    if not match or match.group(1) != expected:
        raise RuntimeError(f"Unexpected uv version: {output.strip()}")
    return output.strip()  # Keep build/platform metadata for provenance.
uv_version = require_uv_version(subprocess.check_output(["uv", "--version"], text=True))
subprocess.run(["git", "clone", REPO_URL, str(REPLAY)], check=True)
subprocess.run(["git", "checkout", "--detach", SOURCE_COMMIT], cwd=REPLAY, check=True)
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPLAY, text=True).strip() == SOURCE_COMMIT
subprocess.run(["uv", "python", "install", "3.12"], check=True)
env = dict(os.environ, UV_PROJECT_ENVIRONMENT=str(REPLAY / ".venv"), UV_PYTHON="3.12")
subprocess.run(["uv", "sync", "--project", str(REPLAY), "--locked", "--group", "dev"], env=env, check=True)
reports = REPLAY / f"reports/runs/{RUN_ID}"
reports.mkdir(parents=True)
def run(*args, name):
    result = subprocess.run(["uv", "run", "--project", str(REPLAY), "--locked", *map(str, args)],
                            cwd=REPLAY, env=env, capture_output=True, text=True, check=False)
    (reports / f"{name}.stdout.log").write_text(result.stdout, encoding="utf-8")
    (reports / f"{name}.stderr.log").write_text(result.stderr, encoding="utf-8")
    print(result.stdout)
    print(result.stderr)
    result.check_returncode()
    return result

preflight = r"""
import hashlib
import json
import shutil
import sys
from pathlib import Path
import yaml
root = Path(sys.argv[1]).resolve()
manifest = yaml.safe_load(Path("data/raw/manifest.yaml").read_text(encoding="utf-8"))
assets = [asset for dataset in manifest["datasets"] for asset in dataset["assets"]]
required = sum(int(asset["bytes"]) for asset in assets) + int(float(sys.argv[2]) * 1024**3)
if shutil.disk_usage(Path.cwd()).free < required:
    raise RuntimeError(f"Insufficient /content scratch space: need {required} bytes")
verified = {}
for asset in assets:
    path = (root / asset["path"]).resolve()
    if not path.is_relative_to(root) or not path.is_file():
        raise ValueError(f"Missing or unsafe manifest asset: {asset['path']}")
    if path.stat().st_size != int(asset["bytes"]):
        raise ValueError(f"Size mismatch: {asset['path']}")
    with path.open("rb") as stream:
        digest = hashlib.file_digest(stream, "sha256").hexdigest()
    if digest != asset["sha256"]:
        raise ValueError(f"SHA-256 mismatch: {asset['path']}")
    verified[asset["path"]] = digest
print(json.dumps({"input_hashes": verified, "required_scratch_bytes": required}))
"""
OUTPUT_RESERVE_GIB = 20  # Match the notebook parameter.
run("python", "-c", preflight, LOCAL_SNAPSHOT / "raw", OUTPUT_RESERVE_GIB, name="preflight")
local = REPLAY / "configs/local"
local.mkdir(parents=True)
(local / "sources.yaml").write_text(json.dumps({"sources": {"raw_root": str(LOCAL_SNAPSHOT / "raw")}}), encoding="utf-8")
reviews = []
for filename, flag in (("review_decisions.csv", "--match-reviews"), ("event_reviews.csv", "--event-reviews")):
    source = LOCAL_SNAPSHOT / "review" / filename
    if source.is_file():
        shutil.copyfile(source, local / filename)
        reviews.extend([flag, f"configs/local/{filename}"])
common = ["--repo-root", str(REPLAY), "--config", "configs/experiments.yaml", "--run-id", RUN_ID]
run("pytest", "tests/integration/test_track_b.py", "-q", name="synthetic_smoke")
run("python", "scripts/run_pipeline.py", *common, *reviews, name="pipeline")
processed = f"data/processed/{RUN_ID}"
links = ["--confirmed-event-segments", f"{processed}/confirmed_event_segments.parquet",
         "--partitions", f"{processed}/road_partitions.parquet"]
run("python", "scripts/train.py", *common, "--features", f"{processed}/road_features.parquet",
    "--labels", f"{processed}/road_labels.parquet", *links, name="train")
track_b = json.loads((REPLAY / f"reports/runs/{RUN_ID}/track_b.json").read_text())
optional = ["--track-b-scores", f"{processed}/track_b_scores.parquet"] if track_b["status"] == "ready" else []
run("python", "scripts/evaluate.py", *common, "--scores", f"{processed}/road_susceptibility.parquet",
    "--roads", f"data/interim/{RUN_ID}/road_segments.gpkg", *links, *optional, name="evaluate")
runtime = json.loads(run("python", "-c",
    "import json, platform; print(json.dumps({'python_version': platform.python_version(), 'platform': platform.platform()}))",
    name="runtime").stdout)
runtime.update({"uv_version": uv_version, "source_commit": SOURCE_COMMIT, "run_id": RUN_ID})
(reports / "runtime.json").write_text(json.dumps(runtime, indent=2), encoding="utf-8")
evaluation = json.loads((reports / "evaluation.json").read_text())
(reports / "colab_summary.md").write_text(
    f"# Local replay {RUN_ID}\n\nSource: {SOURCE_COMMIT}\nTrack B: {track_b['status']}\n"
    f"Evaluation: {evaluation['status']}\n", encoding="utf-8")
run("python", "scripts/verify_colab_run.py", *common,
    "--published-run", str(LOCAL_SNAPSHOT / "runs" / RUN_ID), "--atol", "1e-5",
    "--output", str(REPLAY / f"reports/runs/{RUN_ID}/colab_comparison.json"), name="verify")
```

Inspect the comparison report and exit code. Pending review gates remain pending on both machines; verification does not create evidence or reviews.
